# Sobreajuste y Regularización en MNIST

En este notebook entrenamos 6 clasificadores sobre MNIST con la **misma arquitectura** pero distintas combinaciones de regularización y aumento de datos, para ver cómo cada técnica afecta el sobreajuste.

| # | Modelo | Dropout | Weight Decay | Dataset |
|---|--------|---------|--------------|----------|
| M1 | Base | ✗ | ✗ | Original |
| M2 | Aumentado | ✗ | ✗ | Aumentado |
| M3 | Weight Decay | ✗ | ✓ | Original |
| M4 | Weight Decay + Aug | ✗ | ✓ | Aumentado |
| M5 | Dropout | ✓ | ✗ | Original |
| M6 | Dropout + WD + Aug | ✓ | ✓ | Aumentado |

In [ ]:
!pip install torch torchvision tqdm matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando: {device}')

## Datos

Cargamos MNIST en dos versiones:
- **Original**: normalización estándar
- **Aumentado**: rotaciones (±15°), traslaciones (±10%) y escalado (90–110%)

El conjunto de test **nunca** se aumenta.

In [ ]:
BATCH_SIZE = 64

normalize = transforms.Normalize((0.1307,), (0.3081,))

transform_original = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])

transform_augmented = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.90, 1.10)),
    transforms.ToTensor(),
    normalize,
])

train_original = datasets.MNIST(root='data', train=True,  download=True, transform=transform_original)
train_augmented = datasets.MNIST(root='data', train=True,  download=True, transform=transform_augmented)
test_set       = datasets.MNIST(root='data', train=False, download=True, transform=transform_original)

loader_original  = DataLoader(train_original,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
loader_augmented = DataLoader(train_augmented, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
loader_test      = DataLoader(test_set,        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Visualizar ejemplos originales vs aumentados
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Ejemplos: original (arriba) vs aumentado (abajo)', fontsize=13)
orig_imgs  = [train_original[i][0]  for i in range(8)]
aug_imgs   = [train_augmented[i][0] for i in range(8)]
for col in range(8):
    axes[0, col].imshow(orig_imgs[col].squeeze(), cmap='gray')
    axes[0, col].axis('off')
    axes[1, col].imshow(aug_imgs[col].squeeze(), cmap='gray')
    axes[1, col].axis('off')
plt.tight_layout()
plt.show()

## Arquitectura

Todos los modelos tienen **la misma cantidad de capas y neuronas**:

```
784 → 512 → 256 → 128 → 64 → 10
```

El parámetro `use_dropout` añade una capa `Dropout(0.5)` después de cada ReLU sin cambiar el número de neuronas.

In [ ]:
class MNISTClassifier(nn.Module):
    """Clasificador MLP para MNIST. Misma arquitectura para todos los experimentos."""

    def __init__(self, use_dropout: bool = False):
        super().__init__()
        p = 0.5 if use_dropout else 0.0

        layers = []
        sizes = [784, 512, 256, 128, 64]
        for in_f, out_f in zip(sizes, sizes[1:]):
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.ReLU())
            if use_dropout:
                layers.append(nn.Dropout(p))
        layers.append(nn.Linear(64, 10))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

## Función de entrenamiento

In [ ]:
def train_model(model, train_loader, test_loader, epochs=20, weight_decay=0.0, label='modelo'):
    """Entrena el modelo y devuelve historial de pérdida y accuracy."""
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    train_losses, test_losses = [], []
    train_accs,   test_accs   = [], []

    pbar = tqdm(range(epochs), desc=label, leave=True)
    for epoch in pbar:
        # --- Entrenamiento ---
        model.train(True)
        running_loss = 0.0
        correct, total = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = loss_fn(out, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total   += labels.size(0)
        train_losses.append(running_loss / len(train_loader))
        train_accs.append(correct / total)

        # --- Evaluación en test ---
        model.eval()
        test_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                test_loss += loss_fn(out, labels).item()
                correct   += (out.argmax(1) == labels).sum().item()
                total     += labels.size(0)
        test_losses.append(test_loss / len(test_loader))
        test_accs.append(correct / total)

        pbar.set_postfix({
            'train_loss': f'{train_losses[-1]:.4f}',
            'test_loss':  f'{test_losses[-1]:.4f}',
            'test_acc':   f'{test_accs[-1]:.3f}',
        })

    return {
        'label':        label,
        'train_losses': train_losses,
        'test_losses':  test_losses,
        'train_accs':   train_accs,
        'test_accs':    test_accs,
    }

## M1 — Base (sin regularización, datos originales)

In [ ]:
EPOCHS = 20

m1 = MNISTClassifier(use_dropout=False)
hist_m1 = train_model(m1, loader_original, loader_test, epochs=EPOCHS,
                      weight_decay=0.0, label='M1: Base')

## M2 — Aumento de datos (sin regularización, datos aumentados)

In [ ]:
m2 = MNISTClassifier(use_dropout=False)
hist_m2 = train_model(m2, loader_augmented, loader_test, epochs=EPOCHS,
                      weight_decay=0.0, label='M2: Aumentado')

## M3 — Weight Decay (datos originales)

In [ ]:
m3 = MNISTClassifier(use_dropout=False)
hist_m3 = train_model(m3, loader_original, loader_test, epochs=EPOCHS,
                      weight_decay=1e-4, label='M3: Weight Decay')

## M4 — Weight Decay + Aumento de datos

In [ ]:
m4 = MNISTClassifier(use_dropout=False)
hist_m4 = train_model(m4, loader_augmented, loader_test, epochs=EPOCHS,
                      weight_decay=1e-4, label='M4: WD + Aumentado')

## M5 — Dropout (datos originales)

In [ ]:
m5 = MNISTClassifier(use_dropout=True)
hist_m5 = train_model(m5, loader_original, loader_test, epochs=EPOCHS,
                      weight_decay=0.0, label='M5: Dropout')

## M6 — Dropout + Weight Decay + Aumento de datos

In [ ]:
m6 = MNISTClassifier(use_dropout=True)
hist_m6 = train_model(m6, loader_augmented, loader_test, epochs=EPOCHS,
                      weight_decay=1e-4, label='M6: Dropout + WD + Aumentado')

## Curvas de pérdida

Para cada modelo se muestra la pérdida en entrenamiento (azul) y en test (naranja). Una brecha grande indica sobreajuste.

In [ ]:
all_hists = [hist_m1, hist_m2, hist_m3, hist_m4, hist_m5, hist_m6]
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Curvas de pérdida por modelo', fontsize=15, fontweight='bold')

for ax, hist in zip(axes.flat, all_hists):
    ax.plot(epochs_range, hist['train_losses'], label='Entrenamiento', color='steelblue',  linewidth=2)
    ax.plot(epochs_range, hist['test_losses'],  label='Test',          color='darkorange', linewidth=2)
    ax.set_title(hist['label'], fontsize=12)
    ax.set_xlabel('Época')
    ax.set_ylabel('Pérdida (Cross-Entropy)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Rendimiento final

Comparamos la pérdida y el accuracy en la última época, tanto en entrenamiento como en test.

In [ ]:
# --- Tabla de resultados ---
print(f"{'Modelo':<30} {'Train Loss':>10} {'Test Loss':>10} {'Train Acc':>10} {'Test Acc':>10} {'Gap Acc':>10}")
print('-' * 80)
for h in all_hists:
    tr_loss = h['train_losses'][-1]
    te_loss = h['test_losses'][-1]
    tr_acc  = h['train_accs'][-1]
    te_acc  = h['test_accs'][-1]
    gap     = tr_acc - te_acc
    print(f"{h['label']:<30} {tr_loss:>10.4f} {te_loss:>10.4f} {tr_acc:>10.3f} {te_acc:>10.3f} {gap:>10.3f}")

In [ ]:
# --- Gráfico de barras: accuracy en entrenamiento y test ---
labels_short = ['M1\nBase', 'M2\nAug', 'M3\nWD', 'M4\nWD+Aug', 'M5\nDropout', 'M6\nAll']
train_accs_final = [h['train_accs'][-1] for h in all_hists]
test_accs_final  = [h['test_accs'][-1]  for h in all_hists]

x = np.arange(len(labels_short))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_train = ax.bar(x - width/2, train_accs_final, width, label='Train Acc', color='steelblue',  alpha=0.85)
bars_test  = ax.bar(x + width/2, test_accs_final,  width, label='Test Acc',  color='darkorange', alpha=0.85)

# Anotaciones
for bar in bars_train:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars_test:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Accuracy')
ax.set_title('Accuracy final en entrenamiento y test por modelo', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels_short)
ax.set_ylim(0.9, 1.01)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Brecha entre train acc y test acc (sobreajuste) ---
gaps = [tr - te for tr, te in zip(train_accs_final, test_accs_final)]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if g > 0.01 else '#2ca02c' for g in gaps]
bars = ax.bar(labels_short, gaps, color=colors, alpha=0.85)
for bar, g in zip(bars, gaps):
    ax.annotate(f'{g:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Train Acc − Test Acc')
ax.set_title('Brecha de sobreajuste por modelo\n(rojo = sobreajuste notable, verde = generalización aceptable)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusiones

| Técnica | Efecto esperado |
|---------|----------------|
| **Sin regularización (M1)** | Mayor brecha train/test → sobreajuste |
| **Aumento de datos (M2)** | El modelo ve más variación → mejor generalización |
| **Weight Decay (M3)** | Penaliza pesos grandes → modelos más simples |
| **WD + Aug (M4)** | Combinación de ambas → regularización más fuerte |
| **Dropout (M5)** | Apaga neuronas aleatoriamente → evita co-adaptación |
| **Dropout + WD + Aug (M6)** | Regularización máxima → generalización óptima |

En MNIST el sobreajuste es limitado por la simplicidad del dataset, pero las curvas de pérdida y la brecha train−test ilustran claramente el efecto de cada técnica.

---

# Parte 2: CIFAR-10 — Donde el sobreajuste se pone serio

MNIST es "demasiado fácil" para ver sobreajuste dramático. CIFAR-10 tiene imágenes a color de 32×32 (3072 dimensiones) con 10 clases mucho más complejas. Un MLP sobreparametrizado aquí **memoriza** el dataset fácilmente.

En esta sección:
1. **6 modelos base** (misma estructura que MNIST pero más grande)
2. **Variación del tamaño del dataset** — cómo cambia el sobreajuste con 500, 1k, 5k, 10k, 50k ejemplos
3. **L1 vs L2** — diferencia en las distribuciones de pesos
4. **Early Stopping** — detener el entrenamiento en el momento óptimo
5. **Label Smoothing** — suavizar las etiquetas como regularización
6. **Mixup** — interpolar entre ejemplos de entrenamiento

## Datos CIFAR-10

Usamos un **subconjunto de 5000 ejemplos** para que el sobreajuste sea dramático. La versión aumentada incluye rotaciones, traslaciones, escalado y flip horizontal.

In [ ]:
from torch.utils.data import Subset

CIFAR_SUBSET = 5000
CIFAR_BATCH  = 64
CIFAR_EPOCHS = 40

cifar_normalize = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))

cifar_transform_original = transforms.Compose([
    transforms.ToTensor(),
    cifar_normalize,
])

cifar_transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.90, 1.10)),
    transforms.ToTensor(),
    cifar_normalize,
])

cifar_train_full_orig = datasets.CIFAR10(root='data', train=True,  download=True, transform=cifar_transform_original)
cifar_train_full_aug  = datasets.CIFAR10(root='data', train=True,  download=True, transform=cifar_transform_augmented)
cifar_test_set        = datasets.CIFAR10(root='data', train=False, download=True, transform=cifar_transform_original)

# Subconjunto de 5000 ejemplos (mismos índices para original y aumentado)
torch.manual_seed(42)
subset_indices = torch.randperm(len(cifar_train_full_orig))[:CIFAR_SUBSET].tolist()

cifar_train_orig = Subset(cifar_train_full_orig, subset_indices)
cifar_train_aug  = Subset(cifar_train_full_aug,  subset_indices)

cifar_loader_orig = DataLoader(cifar_train_orig, batch_size=CIFAR_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
cifar_loader_aug  = DataLoader(cifar_train_aug,  batch_size=CIFAR_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
cifar_loader_test = DataLoader(cifar_test_set,   batch_size=CIFAR_BATCH, shuffle=False, num_workers=2, pin_memory=True)

# Visualizar
cifar_classes = ['avión', 'auto', 'pájaro', 'gato', 'ciervo', 'perro', 'rana', 'caballo', 'barco', 'camión']
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('CIFAR-10: original (arriba) vs aumentado (abajo)', fontsize=13)
for col in range(8):
    img_o, lbl = cifar_train_orig[col]
    img_a, _   = cifar_train_aug[col]
    # Desnormalizar para visualización
    for img, ax_row in [(img_o, 0), (img_a, 1)]:
        img_show = img.permute(1, 2, 0).numpy()
        img_show = img_show * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
        img_show = np.clip(img_show, 0, 1)
        axes[ax_row, col].imshow(img_show)
        axes[ax_row, col].axis('off')
    axes[0, col].set_title(cifar_classes[lbl], fontsize=9)
plt.tight_layout()
plt.show()
print(f'Tamaño del subconjunto de entrenamiento: {len(cifar_train_orig)}')

## Arquitectura CIFAR-10

MLP **sobreparametrizado** para que el sobreajuste sea evidente:

```
3072 → 2048 → 2048 → 1024 → 512 → 10
```

Con ~12M de parámetros y solo 5000 ejemplos, este modelo puede memorizar todo el dataset.

In [ ]:
class CIFARClassifier(nn.Module):
    """MLP sobreparametrizado para CIFAR-10."""

    def __init__(self, use_dropout: bool = False, dropout_p: float = 0.5):
        super().__init__()
        layers = []
        sizes = [3072, 2048, 2048, 1024, 512]
        for in_f, out_f in zip(sizes, sizes[1:]):
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.ReLU())
            if use_dropout:
                layers.append(nn.Dropout(dropout_p))
        layers.append(nn.Linear(512, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

# Contar parámetros
_tmp = CIFARClassifier()
n_params = sum(p.numel() for p in _tmp.parameters())
print(f'Parámetros del MLP: {n_params:,} ({n_params/1e6:.1f}M)')
print(f'Ratio parámetros/ejemplos: {n_params/CIFAR_SUBSET:.0f}:1 — sobreajuste garantizado')
del _tmp

## Función de entrenamiento CIFAR-10

Versión extendida que soporta todas las técnicas de regularización:
- **Weight Decay (L2)** — vía el optimizador
- **L1** — penalización manual sobre los pesos
- **Label Smoothing** — suaviza las etiquetas one-hot
- **Mixup** — interpola entre pares de ejemplos
- **Early Stopping** — detiene el entrenamiento si el test loss no mejora

In [ ]:
import copy

def train_cifar(model, train_loader, test_loader, epochs=40, weight_decay=0.0,
                l1_lambda=0.0, label_smoothing=0.0, mixup_alpha=0.0,
                early_stopping_patience=0, label='modelo'):
    """
    Entrena un modelo CIFAR-10 con soporte para múltiples técnicas de regularización.

    Args:
        l1_lambda: coeficiente de regularización L1 (0 = desactivado)
        label_smoothing: suavizado de etiquetas (0 = desactivado, típico: 0.1)
        mixup_alpha: parámetro alpha de mixup (0 = desactivado, típico: 0.2-0.4)
        early_stopping_patience: épocas sin mejora antes de parar (0 = desactivado)
    """
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    train_losses, test_losses = [], []
    train_accs,   test_accs   = [], []

    best_test_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    stopped_epoch = epochs

    pbar = tqdm(range(epochs), desc=label, leave=True)
    for epoch in pbar:
        # --- Entrenamiento ---
        model.train(True)
        running_loss = 0.0
        correct, total = 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            # Mixup
            if mixup_alpha > 0:
                lam = np.random.beta(mixup_alpha, mixup_alpha)
                idx = torch.randperm(imgs.size(0), device=device)
                imgs_mixed = lam * imgs + (1 - lam) * imgs[idx]
                optimizer.zero_grad()
                out = model(imgs_mixed)
                loss = lam * loss_fn(out, labels) + (1 - lam) * loss_fn(out, labels[idx])
            else:
                optimizer.zero_grad()
                out = model(imgs)
                loss = loss_fn(out, labels)

            # Regularización L1
            if l1_lambda > 0:
                l1_norm = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_norm

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total   += labels.size(0)

        train_losses.append(running_loss / len(train_loader))
        train_accs.append(correct / total)

        # --- Evaluación en test ---
        model.eval()
        test_loss = 0.0
        correct, total = 0, 0
        eval_loss_fn = nn.CrossEntropyLoss()  # Sin label smoothing para evaluar
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                test_loss += eval_loss_fn(out, labels).item()
                correct   += (out.argmax(1) == labels).sum().item()
                total     += labels.size(0)
        test_losses.append(test_loss / len(test_loader))
        test_accs.append(correct / total)

        pbar.set_postfix({
            'tr_loss': f'{train_losses[-1]:.4f}',
            'te_loss': f'{test_losses[-1]:.4f}',
            'te_acc':  f'{test_accs[-1]:.3f}',
        })

        # Early stopping
        if early_stopping_patience > 0:
            if test_losses[-1] < best_test_loss:
                best_test_loss = test_losses[-1]
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= early_stopping_patience:
                    stopped_epoch = epoch + 1
                    print(f'  → Early stopping en época {stopped_epoch} (sin mejora durante {early_stopping_patience} épocas)')
                    model.load_state_dict(best_model_state)
                    break

    return {
        'label':         label,
        'train_losses':  train_losses,
        'test_losses':   test_losses,
        'train_accs':    train_accs,
        'test_accs':     test_accs,
        'stopped_epoch': stopped_epoch,
        'model':         model,
    }

## Experimento 1: Los 6 modelos base en CIFAR-10

Misma estructura que la parte de MNIST pero con el MLP sobreparametrizado y solo 5000 ejemplos.

In [ ]:
# C1: Base — sin regularización, datos originales
c1 = CIFARClassifier(use_dropout=False)
hist_c1 = train_cifar(c1, cifar_loader_orig, cifar_loader_test, epochs=CIFAR_EPOCHS,
                      label='C1: Base')

In [ ]:
# C2: Aumento de datos — sin regularización, datos aumentados
c2 = CIFARClassifier(use_dropout=False)
hist_c2 = train_cifar(c2, cifar_loader_aug, cifar_loader_test, epochs=CIFAR_EPOCHS,
                      label='C2: Aumentado')

In [ ]:
# C3: Weight Decay — datos originales
c3 = CIFARClassifier(use_dropout=False)
hist_c3 = train_cifar(c3, cifar_loader_orig, cifar_loader_test, epochs=CIFAR_EPOCHS,
                      weight_decay=1e-4, label='C3: Weight Decay')

In [ ]:
# C4: Weight Decay + Aumento de datos
c4 = CIFARClassifier(use_dropout=False)
hist_c4 = train_cifar(c4, cifar_loader_aug, cifar_loader_test, epochs=CIFAR_EPOCHS,
                      weight_decay=1e-4, label='C4: WD + Aumentado')

In [ ]:
# C5: Dropout — datos originales
c5 = CIFARClassifier(use_dropout=True)
hist_c5 = train_cifar(c5, cifar_loader_orig, cifar_loader_test, epochs=CIFAR_EPOCHS,
                      label='C5: Dropout')

In [ ]:
# C6: Dropout + Weight Decay + Aumento de datos
c6 = CIFARClassifier(use_dropout=True)
hist_c6 = train_cifar(c6, cifar_loader_aug, cifar_loader_test, epochs=CIFAR_EPOCHS,
                      weight_decay=1e-4, label='C6: Dropout + WD + Aumentado')

In [ ]:
cifar_base_hists = [hist_c1, hist_c2, hist_c3, hist_c4, hist_c5, hist_c6]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('CIFAR-10: Curvas de pérdida (5000 ejemplos, MLP sobreparametrizado)', fontsize=15, fontweight='bold')

for ax, hist in zip(axes.flat, cifar_base_hists):
    ep = range(1, len(hist['train_losses']) + 1)
    ax.plot(ep, hist['train_losses'], label='Entrenamiento', color='steelblue',  linewidth=2)
    ax.plot(ep, hist['test_losses'],  label='Test',          color='darkorange', linewidth=2)
    ax.set_title(hist['label'], fontsize=12)
    ax.set_xlabel('Época')
    ax.set_ylabel('Pérdida')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tabla y barras de los 6 modelos base CIFAR-10
print(f"{'Modelo':<35} {'Train Loss':>10} {'Test Loss':>10} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>8}")
print('-' * 83)
for h in cifar_base_hists:
    tr_l, te_l = h['train_losses'][-1], h['test_losses'][-1]
    tr_a, te_a = h['train_accs'][-1],   h['test_accs'][-1]
    print(f"{h['label']:<35} {tr_l:>10.4f} {te_l:>10.4f} {tr_a:>10.3f} {te_a:>10.3f} {tr_a-te_a:>8.3f}")

# Gráfico de barras
labels_c = ['C1\nBase', 'C2\nAug', 'C3\nWD', 'C4\nWD+Aug', 'C5\nDropout', 'C6\nAll']
tr_acc_c = [h['train_accs'][-1] for h in cifar_base_hists]
te_acc_c = [h['test_accs'][-1]  for h in cifar_base_hists]

x = np.arange(len(labels_c))
width = 0.35
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.bar(x - width/2, tr_acc_c, width, label='Train', color='steelblue',  alpha=0.85)
ax1.bar(x + width/2, te_acc_c, width, label='Test',  color='darkorange', alpha=0.85)
for i in range(len(labels_c)):
    ax1.annotate(f'{te_acc_c[i]:.3f}', xy=(x[i]+width/2, te_acc_c[i]), xytext=(0,3),
                 textcoords='offset points', ha='center', fontsize=9)
ax1.set_xticks(x); ax1.set_xticklabels(labels_c)
ax1.set_ylabel('Accuracy'); ax1.set_title('Accuracy final', fontweight='bold')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

gaps_c = [tr - te for tr, te in zip(tr_acc_c, te_acc_c)]
colors_c = ['#d62728' if g > 0.05 else '#2ca02c' for g in gaps_c]
bars = ax2.bar(labels_c, gaps_c, color=colors_c, alpha=0.85)
for bar, g in zip(bars, gaps_c):
    ax2.annotate(f'{g:.3f}', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                 xytext=(0,3), textcoords='offset points', ha='center', fontsize=10)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Train Acc − Test Acc'); ax2.set_title('Brecha de sobreajuste', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Experimento 2: Efecto del tamaño del dataset

Entrenamos el mismo MLP (sin regularización) variando la cantidad de ejemplos: 500, 1000, 5000, 10000 y 50000. Esto muestra cómo el sobreajuste depende del ratio parámetros/datos.

In [ ]:
data_sizes = [500, 1000, 5000, 10000, 50000]
hist_sizes = {}

torch.manual_seed(42)
all_indices = torch.randperm(len(cifar_train_full_orig)).tolist()

for n in data_sizes:
    indices = all_indices[:n]
    subset = Subset(cifar_train_full_orig, indices)
    loader = DataLoader(subset, batch_size=CIFAR_BATCH, shuffle=True, num_workers=2, pin_memory=True)

    model = CIFARClassifier(use_dropout=False)
    hist_sizes[n] = train_cifar(model, loader, cifar_loader_test, epochs=CIFAR_EPOCHS,
                                label=f'N={n}')

In [ ]:
# Curvas de pérdida por tamaño de dataset
fig, axes = plt.subplots(1, len(data_sizes), figsize=(18, 4))
fig.suptitle('Efecto del tamaño del dataset (sin regularización)', fontsize=14, fontweight='bold')

for ax, n in zip(axes, data_sizes):
    h = hist_sizes[n]
    ep = range(1, len(h['train_losses']) + 1)
    ax.plot(ep, h['train_losses'], label='Train', color='steelblue',  linewidth=2)
    ax.plot(ep, h['test_losses'],  label='Test',  color='darkorange', linewidth=2)
    ax.set_title(f'N = {n:,}', fontsize=11)
    ax.set_xlabel('Época')
    if n == data_sizes[0]:
        ax.set_ylabel('Pérdida')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Resumen: test accuracy vs tamaño
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
test_accs_by_size  = [hist_sizes[n]['test_accs'][-1]  for n in data_sizes]
train_accs_by_size = [hist_sizes[n]['train_accs'][-1] for n in data_sizes]
gaps_by_size = [tr - te for tr, te in zip(train_accs_by_size, test_accs_by_size)]

ax1.plot(data_sizes, train_accs_by_size, 'o-', color='steelblue',  label='Train Acc', linewidth=2, markersize=8)
ax1.plot(data_sizes, test_accs_by_size,  'o-', color='darkorange', label='Test Acc',  linewidth=2, markersize=8)
ax1.set_xscale('log')
ax1.set_xlabel('Tamaño del dataset (log)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs tamaño del dataset', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(data_sizes, gaps_by_size, 's-', color='#d62728', linewidth=2, markersize=8)
ax2.set_xscale('log')
ax2.set_xlabel('Tamaño del dataset (log)')
ax2.set_ylabel('Train Acc − Test Acc')
ax2.set_title('Brecha de sobreajuste vs datos', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Experimento 3: L1 vs L2 (Lasso vs Ridge)

L1 produce **pesos dispersos** (muchos exactamente cero), mientras que L2 produce **pesos pequeños pero no nulos**. Entrenamos ambos y comparamos la distribución de pesos.

In [ ]:
# Sin regularización (referencia)
model_noreg = CIFARClassifier()
hist_noreg = train_cifar(model_noreg, cifar_loader_orig, cifar_loader_test,
                         epochs=CIFAR_EPOCHS, label='Sin regularización')

# L2 (Weight Decay)
model_l2 = CIFARClassifier()
hist_l2 = train_cifar(model_l2, cifar_loader_orig, cifar_loader_test,
                      epochs=CIFAR_EPOCHS, weight_decay=1e-3, label='L2 (WD=1e-3)')

# L1
model_l1 = CIFARClassifier()
hist_l1 = train_cifar(model_l1, cifar_loader_orig, cifar_loader_test,
                      epochs=CIFAR_EPOCHS, l1_lambda=1e-5, label='L1 (λ=1e-5)')

In [ ]:
# Comparar distribuciones de pesos
def get_all_weights(model):
    """Extrae todos los pesos (no bias) del modelo como un vector plano."""
    weights = []
    for name, param in model.named_parameters():
        if 'weight' in name:
            weights.append(param.detach().cpu().numpy().flatten())
    return np.concatenate(weights)

w_noreg = get_all_weights(hist_noreg['model'])
w_l2    = get_all_weights(hist_l2['model'])
w_l1    = get_all_weights(hist_l1['model'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Distribución de pesos: Sin reg vs L2 vs L1', fontsize=14, fontweight='bold')

for ax, w, title, color in zip(axes,
                                [w_noreg, w_l2, w_l1],
                                ['Sin regularización', 'L2 (Weight Decay)', 'L1 (Lasso)'],
                                ['gray', 'steelblue', 'darkorange']):
    ax.hist(w, bins=100, alpha=0.8, color=color, density=True)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Valor del peso')
    ax.set_ylabel('Densidad')
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlim(-0.3, 0.3)

    # Estadísticas
    pct_near_zero = np.mean(np.abs(w) < 1e-3) * 100
    ax.text(0.95, 0.95, f'|w|<0.001: {pct_near_zero:.1f}%\nstd: {np.std(w):.4f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Curvas de loss comparadas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for h, c in [(hist_noreg, 'gray'), (hist_l2, 'steelblue'), (hist_l1, 'darkorange')]:
    ep = range(1, len(h['train_losses'])+1)
    ax1.plot(ep, h['train_losses'], '--', color=c, alpha=0.6)
    ax1.plot(ep, h['test_losses'],  '-',  color=c, label=h['label'], linewidth=2)
    ax2.plot(ep, h['test_accs'],    '-',  color=c, label=h['label'], linewidth=2)

ax1.set_xlabel('Época'); ax1.set_ylabel('Pérdida')
ax1.set_title('Pérdida test (línea) y train (punteada)', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_xlabel('Época'); ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy: L1 vs L2', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Experimento 4: Early Stopping

La forma más simple de regularización: dejar de entrenar cuando el test loss deja de mejorar. Comparamos el modelo base (que sobreajusta) con uno que usa early stopping (paciencia = 5 épocas).

In [ ]:
# Sin early stopping (referencia, ya tenemos hist_c1 pero reentrenamos para comparar limpiamente)
model_no_es = CIFARClassifier()
hist_no_es = train_cifar(model_no_es, cifar_loader_orig, cifar_loader_test,
                         epochs=CIFAR_EPOCHS, label='Sin Early Stopping')

# Con early stopping (paciencia = 5)
model_es = CIFARClassifier()
hist_es = train_cifar(model_es, cifar_loader_orig, cifar_loader_test,
                      epochs=CIFAR_EPOCHS, early_stopping_patience=5,
                      label='Early Stopping (p=5)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Early Stopping: evitar el sobreajuste parando a tiempo', fontsize=14, fontweight='bold')

for h, c, ls in [(hist_no_es, '#d62728', '-'), (hist_es, '#2ca02c', '-')]:
    ep = range(1, len(h['train_losses'])+1)
    ax1.plot(ep, h['train_losses'], '--', color=c, alpha=0.5)
    ax1.plot(ep, h['test_losses'],  ls,   color=c, label=h['label'], linewidth=2)

# Marcar dónde paró el early stopping
if hist_es['stopped_epoch'] < CIFAR_EPOCHS:
    ax1.axvline(hist_es['stopped_epoch'], color='#2ca02c', linestyle=':', alpha=0.8, linewidth=2)
    ax1.annotate(f'Stop: época {hist_es["stopped_epoch"]}',
                 xy=(hist_es['stopped_epoch'], ax1.get_ylim()[1]*0.9),
                 fontsize=10, color='#2ca02c', fontweight='bold')

ax1.set_xlabel('Época'); ax1.set_ylabel('Pérdida')
ax1.set_title('Pérdida test (línea) y train (punteada)', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

for h, c in [(hist_no_es, '#d62728'), (hist_es, '#2ca02c')]:
    ep = range(1, len(h['test_accs'])+1)
    ax2.plot(ep, h['test_accs'], '-', color=c, label=h['label'], linewidth=2)

ax2.set_xlabel('Época'); ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSin Early Stopping — Test Acc final: {hist_no_es['test_accs'][-1]:.4f} (época {CIFAR_EPOCHS})")
print(f"Con Early Stopping — Test Acc final: {hist_es['test_accs'][-1]:.4f} (época {hist_es['stopped_epoch']})")

---

## Experimento 5: Label Smoothing

En vez de etiquetas duras (0 o 1), usamos etiquetas suaves (ej. 0.9 para la clase correcta, 0.01 para las demás). Esto evita que el modelo se vuelva sobreconfiado y actúa como regularización.

In [ ]:
# Sin label smoothing (referencia)
model_no_ls = CIFARClassifier()
hist_no_ls = train_cifar(model_no_ls, cifar_loader_orig, cifar_loader_test,
                         epochs=CIFAR_EPOCHS, label='Sin Label Smoothing')

# Con label smoothing = 0.1
model_ls = CIFARClassifier()
hist_ls = train_cifar(model_ls, cifar_loader_orig, cifar_loader_test,
                      epochs=CIFAR_EPOCHS, label_smoothing=0.1,
                      label='Label Smoothing (ε=0.1)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Label Smoothing: suavizar etiquetas como regularización', fontsize=14, fontweight='bold')

for h, c in [(hist_no_ls, '#d62728'), (hist_ls, '#2ca02c')]:
    ep = range(1, len(h['train_losses'])+1)
    ax1.plot(ep, h['test_losses'], '-', color=c, label=h['label'], linewidth=2)
    ax1.plot(ep, h['train_losses'], '--', color=c, alpha=0.5)

ax1.set_xlabel('Época'); ax1.set_ylabel('Pérdida')
ax1.set_title('Pérdida test (línea) y train (punteada)', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

for h, c in [(hist_no_ls, '#d62728'), (hist_ls, '#2ca02c')]:
    ep = range(1, len(h['test_accs'])+1)
    ax2.plot(ep, h['test_accs'], '-', color=c, label=h['label'], linewidth=2)

ax2.set_xlabel('Época'); ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Comparar la confianza del modelo (máximo softmax en test)
from torch.nn.functional import softmax

def get_confidences(model, loader):
    model.eval()
    confs = []
    with torch.no_grad():
        for imgs, _ in loader:
            out = model(imgs.to(device))
            probs = softmax(out, dim=1)
            confs.extend(probs.max(dim=1).values.cpu().numpy())
    return np.array(confs)

conf_no_ls = get_confidences(hist_no_ls['model'], cifar_loader_test)
conf_ls    = get_confidences(hist_ls['model'],    cifar_loader_test)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(conf_no_ls, bins=50, alpha=0.6, color='#d62728', label=f'Sin LS (media={conf_no_ls.mean():.3f})', density=True)
ax.hist(conf_ls,    bins=50, alpha=0.6, color='#2ca02c', label=f'Con LS (media={conf_ls.mean():.3f})',    density=True)
ax.set_xlabel('Confianza máxima (softmax)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de confianza en test\n(Label Smoothing reduce la sobreconfianza)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Experimento 6: Mixup

Mixup crea nuevos ejemplos interpolando entre pares de imágenes y sus etiquetas: `x̃ = λx_i + (1-λ)x_j`. Esto suaviza los límites de decisión y actúa como una forma de aumento de datos + regularización.

In [ ]:
# Sin mixup (referencia)
model_no_mix = CIFARClassifier()
hist_no_mix = train_cifar(model_no_mix, cifar_loader_orig, cifar_loader_test,
                          epochs=CIFAR_EPOCHS, label='Sin Mixup')

# Con mixup (alpha=0.3)
model_mix = CIFARClassifier()
hist_mix = train_cifar(model_mix, cifar_loader_orig, cifar_loader_test,
                       epochs=CIFAR_EPOCHS, mixup_alpha=0.3, label='Mixup (α=0.3)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mixup: interpolar entre ejemplos como regularización', fontsize=14, fontweight='bold')

for h, c in [(hist_no_mix, '#d62728'), (hist_mix, '#2ca02c')]:
    ep = range(1, len(h['train_losses'])+1)
    ax1.plot(ep, h['test_losses'], '-', color=c, label=h['label'], linewidth=2)
    ax1.plot(ep, h['train_losses'], '--', color=c, alpha=0.5)

ax1.set_xlabel('Época'); ax1.set_ylabel('Pérdida')
ax1.set_title('Pérdida test (línea) y train (punteada)', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

for h, c in [(hist_no_mix, '#d62728'), (hist_mix, '#2ca02c')]:
    ep = range(1, len(h['test_accs'])+1)
    ax2.plot(ep, h['test_accs'], '-', color=c, label=h['label'], linewidth=2)

ax2.set_xlabel('Época'); ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Visualizar qué aspecto tiene mixup
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Mixup: imágenes originales (arriba) vs interpoladas (abajo)', fontsize=13)
batch_imgs, batch_lbls = next(iter(cifar_loader_orig))
for col in range(8):
    # Original
    img = batch_imgs[col].permute(1,2,0).numpy()
    img = img * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
    axes[0, col].imshow(np.clip(img, 0, 1))
    axes[0, col].set_title(cifar_classes[batch_lbls[col]], fontsize=8)
    axes[0, col].axis('off')
    # Mixup
    lam = np.random.beta(0.3, 0.3)
    j = (col + 4) % len(batch_imgs)
    mixed = lam * batch_imgs[col] + (1 - lam) * batch_imgs[j]
    img_m = mixed.permute(1,2,0).numpy()
    img_m = img_m * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
    axes[1, col].imshow(np.clip(img_m, 0, 1))
    axes[1, col].set_title(f'λ={lam:.2f}', fontsize=8)
    axes[1, col].axis('off')
plt.tight_layout()
plt.show()

---

## Comparación final: todas las técnicas en CIFAR-10

Juntamos los mejores resultados de cada técnica para ver cuál tiene mayor impacto sobre el sobreajuste.

In [ ]:
# Recopilar todos los experimentos
all_cifar = [
    hist_c1,      # Base
    hist_c2,      # Aumentado
    hist_c3,      # Weight Decay
    hist_c5,      # Dropout
    hist_c6,      # Dropout + WD + Aug
    hist_es,      # Early Stopping
    hist_l1,      # L1
    hist_l2,      # L2
    hist_ls,      # Label Smoothing
    hist_mix,     # Mixup
]

# Tabla resumen
print(f"{'Modelo':<35} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>8} {'Épocas':>8}")
print('=' * 71)
for h in all_cifar:
    tr_a, te_a = h['train_accs'][-1], h['test_accs'][-1]
    ep = h.get('stopped_epoch', len(h['train_accs']))
    print(f"{h['label']:<35} {tr_a:>10.3f} {te_a:>10.3f} {tr_a-te_a:>8.3f} {ep:>8}")

In [ ]:
# Gráfico comparativo final
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9))

names  = [h['label'] for h in all_cifar]
tr_acc = [h['train_accs'][-1] for h in all_cifar]
te_acc = [h['test_accs'][-1]  for h in all_cifar]
gaps   = [tr - te for tr, te in zip(tr_acc, te_acc)]

# Ordenar por test accuracy descendente
order = np.argsort(te_acc)[::-1]
names_s  = [names[i] for i in order]
tr_acc_s = [tr_acc[i] for i in order]
te_acc_s = [te_acc[i] for i in order]
gaps_s   = [gaps[i] for i in order]

x = np.arange(len(names_s))
width = 0.35

# Accuracy
ax1.barh(x - width/2, tr_acc_s, width, label='Train Acc', color='steelblue',  alpha=0.85)
ax1.barh(x + width/2, te_acc_s, width, label='Test Acc',  color='darkorange', alpha=0.85)
for i in range(len(names_s)):
    ax1.annotate(f'{te_acc_s[i]:.3f}', xy=(te_acc_s[i], x[i]+width/2),
                 xytext=(5, 0), textcoords='offset points', va='center', fontsize=9)
ax1.set_yticks(x); ax1.set_yticklabels(names_s)
ax1.set_xlabel('Accuracy')
ax1.set_title('Accuracy final por técnica de regularización (ordenado por Test Acc)', fontsize=13, fontweight='bold')
ax1.legend(loc='lower right'); ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

# Brecha
colors_f = ['#d62728' if g > 0.1 else '#ff7f0e' if g > 0.05 else '#2ca02c' for g in gaps_s]
ax2.barh(x, gaps_s, color=colors_f, alpha=0.85)
for i, g in enumerate(gaps_s):
    ax2.annotate(f'{g:.3f}', xy=(g, x[i]), xytext=(5, 0),
                 textcoords='offset points', va='center', fontsize=9)
ax2.set_yticks(x); ax2.set_yticklabels(names_s)
ax2.set_xlabel('Train Acc − Test Acc')
ax2.set_title('Brecha de sobreajuste (rojo=severo, naranja=moderado, verde=aceptable)',
              fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## Conclusiones CIFAR-10

### Las 6 técnicas de regularización estudiadas

| Técnica | Qué hace | Efecto principal |
|---------|----------|-----------------|
| **Data Augmentation** | Genera variaciones de los datos existentes | Más datos efectivos → reduce sobreajuste |
| **Weight Decay (L2)** | Penaliza pesos grandes: `loss += λΣw²` | Pesos pequeños → modelo más simple |
| **L1 (Lasso)** | Penaliza la magnitud absoluta: `loss += λΣ|w|` | Pesos dispersos → selección de features |
| **Dropout** | Apaga neuronas al azar durante entrenamiento | Evita co-adaptación → ensemble implícito |
| **Early Stopping** | Para el entrenamiento cuando el test loss sube | Evita memorización → zero cost |
| **Label Smoothing** | Suaviza etiquetas: `[0,1,0]` → `[0.05, 0.9, 0.05]` | Reduce sobreconfianza → mejor calibración |
| **Mixup** | Interpola entre pares de ejemplos y etiquetas | Suaviza límites de decisión → mejor generalización |

### Lecciones clave

1. **Más datos > más regularización**: El aumento de datos suele tener el mayor impacto porque ataca la raíz del problema (pocos datos).

2. **Las técnicas son complementarias**: Combinar dropout + weight decay + aumento de datos funciona mejor que cualquiera por separado.

3. **El sobreajuste escala con el ratio parámetros/datos**: Con 500 ejemplos el modelo memoriza todo; con 50,000 el sobreajuste se reduce naturalmente.

4. **Early stopping es gratis**: No cambia la arquitectura ni el entrenamiento, solo decide cuándo parar.

5. **L1 vs L2 tienen efectos distintos**: L1 produce pesos exactamente cero (selección de features), L2 produce pesos uniformemente pequeños.

6. **CIFAR-10 con MLP muestra el sobreajuste mucho más claramente que MNIST**, porque el problema es más difícil y el MLP no tiene las inductive biases correctas para imágenes.